In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [4]:
# 1. 准备数据（注意：CNN需要4维输入：样本数，长，宽，通道数）
mnist = tf.keras.datasets.mnist
(x_train,y_train),(x_test,y_test) = mnist.load_data()

#归一化并调整形状（28,28）->(28,28,1)
x_train = x_train.reshape((60000,28,28,1)).astype('float32') / 255
x_test = x_test.reshape((10000,28,28,1)).astype('float32') / 255

In [7]:
# 2.搭建CNN模型
model = models.Sequential([
    #卷积层： 32 个 3*3的核，使用Padding保持尺寸不变（padding=‘same’）
    layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(28,28,1)),
    #池化层:把28 * 28 压缩成14 * 14
    layers.MaxPooling2D((2,2)),
    
    #再来一层卷积提取更深特征
    layers.Conv2D(64,(3,3),activation='relu'),
    #再池化：把14 * 14 压缩成 7 *7
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),    #拍扁
    layers.Dense(64, activation='relu'),  #全连接
    layers.Dense(10, activation='softmax')  #输出10个数字的概率
    
])


In [9]:
# 3.编译并训练
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=64)


Epoch 1/5


938/938 [==============================] - 8s 8ms/step - loss: 0.1780 - accuracy: 0.9473
Epoch 2/5
938/938 [==============================] - 7s 7ms/step - loss: 0.0534 - accuracy: 0.9840
Epoch 3/5
938/938 [==============================] - 7s 7ms/step - loss: 0.0358 - accuracy: 0.9888
Epoch 4/5
938/938 [==============================] - 7s 7ms/step - loss: 0.0270 - accuracy: 0.9916
Epoch 5/5
938/938 [==============================] - 7s 7ms/step - loss: 0.0215 - accuracy: 0.9934


In [10]:
# 4.评估结果
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"CNN 测试准确率：{test_acc}")

313/313 [==============================] - 1s 2ms/step - loss: 0.0339 - accuracy: 0.9886
CNN 测试准确率：0.9886000156402588


📑 知识点应试八股文总结（Day 5 - 完结篇）
 1. 核心定义激活函数 (Activation Function) 
是神经网络中负责引入非线性因素的关键组件。如果没有激活函数，多层神经网络在数学上等价于单层感知机。
2. 考点拆解线性坍塌：若无非线性激活，$f(f(x)) = W_2(W_1x) = (W_2W_1)x = W'x$。无论层数多深，表达能力依然受限。ReLU 的优势：计算简单：只需比较大小，不像 Sigmoid 需要算指数。缓解梯度消失：在 $x > 0$ 区域导数为 $1$，信号传输不衰减。卷积层的参数计算：参数量 = $(F \times F \times C_{in} + 1) \times C_{out}$（其中 $F$ 是核大小，$C_{in}$ 是输入通道，$+1$ 是偏置，$C_{out}$ 是卷积核个数）。
3. 高频面试问答
Q：ReLU 在 $x < 0$ 时梯度为 0，这会导致什么问题？
A：会导致 Dead ReLU（神经元坏死） 现象。如果一个神经元的输出长期为负，它将永远无法被更新。解决方法包括使用 Leaky ReLU（给负半轴一点点斜率）或减小学习率。

Q：既然 CNN 这么强，它有什么缺点吗？
A：1. 缺乏全局建模能力（只看局部，虽然可以通过堆叠多层扩大感受野）；
2. 对旋转比较敏感（除非数据增强，否则旋转 90 度的猫它可能不认识）。

4. 速记要点非线性是灵魂，ReLU 是刀，剪掉负数出曲线。卷积计算公式必背：$(W - F + 2P)/S + 1$